<img src="logo.png" alt="Vegeta" width="240">

# Myropods: Apheloria — the modular walker that rolls into a ball

The pill millipede of the family: large modular segments (a 110 mm body cylinder, 150 mm long, 0.8–1 kg,
four legs each), a sensor head, and an armour plate on every back that is a wedge of **one sphere** —
curl the chain nose-under with 360°/n at every joint and the plates close into a ball (*glomeris mode*:
protection, low profile, hibernation, rolling). Segments are modules (camera, lidar, battery, AI core,
sampler…) that connect and lock, so the same design file builds a 4-segment *mini* and a 14-segment
*max* configuration (`designs/apheloria.py`).

```
CAD (Dedalus): segment + plate, leg, head; the chain in walk mode and in ball mode ─► masses per configuration against the sheet
ball geometry vs segment count ─► rolling (slope to start, speed, kinetic energy) ─► a drop in ball mode ─► plate FEA (Talos)
curling up ─► the joint lifting half the chain ─► joint actuator class
walking at 6 cm/s ─► foot forces ─► leg servo torque ─► leg FEA ─► joint neck FEA (the chain hanging in a curl)
battery modules ─► endurance per mode ─► JSON record
```

In [ ]:
# DEBUG: test the notebook end to end before the full run. On: FEA and CFD are mocked (vegeta.mock: fixed numbers,
# seconds instead of hours, the result cache off). Turn it on here or from the shell:
#   VEGETA_DEBUG=1 jupyter nbconvert --to notebook --execute <notebook>.ipynb --output <notebook>-run.ipynb
DEBUG = False
from vegeta import mock
DEBUG = mock.setup(DEBUG)

In [ ]:
import json, math, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from vegeta import dedalus, talos
from vegeta.dedalus import viz as dviz
from vegeta.talos import viz as tviz
import sys; sys.path.insert(0, "designs"); import actuators as act          # the shared actuator / motor / joint library (designs/actuators.py)

RUNS = Path("_runs/apheloria"); shutil.rmtree(RUNS, ignore_errors=True); RUNS.mkdir(parents=True)
design_file = RUNS / "apheloria.py"
shutil.copy(Path("designs/apheloria.py"), design_file)
aph = dedalus.load_design(f"{design_file}:Apheloria")
pd.DataFrame(aph.params.table()).set_index("name")

## 1. Walk mode and ball mode

In [ ]:
p = aph.resolve()
walk = aph.generate(mode="walk"); ball = aph.generate(mode="ball")
seg, plate, leg, head = aph.generate(part="segment"), aph.generate(part="plate"), aph.generate(part="leg"), aph.generate(part="head")
cad = {k: g.export(RUNS / "cad" / k, stl_tolerance=0.05) for k, g in (("walk", walk), ("ball", ball), ("segment", seg), ("plate", plate), ("leg", leg), ("head", head))}
N = p["n_segments"]; REACH = aph.leg_reach(p)
print(f"{N} segments + head | walk mode {walk.dimensions[0]:.0f} x {walk.dimensions[1]:.0f} x {walk.dimensions[2]:.0f} mm | ball {ball.dimensions[0]:.0f} x {ball.dimensions[1]:.0f} x {ball.dimensions[2]:.0f} mm | coil radius {aph.coil_radius(p):.0f} mm, ball radius {aph.ball_radius(p):.0f} mm, {aph.coil_angle_deg(p):.0f}° per joint")
dviz.show(dviz.plot3d(walk))

In [ ]:
dviz.show(dviz.plot3d(ball))

In [ ]:
fig = dviz.plot_sections(seg, normal="x", positions=[0.0, aph.hip_x(p)], cols=2)
fig = dviz.plot_sections(ball, normal="y", positions=[0.0], cols=1)

In [ ]:
RHO_PA12CF, RHO_AL = 1.1e-3, 2.7e-3
ACT = {"leg": act.get("smart servo 12 Nm").as_dict(), "joint": act.get("qdd 60 Nm").as_dict()}      # from the shared library
base_segment = pd.Series({"body shell (CAD, PA12-CF)": (seg.volume - plate.volume) * RHO_PA12CF, "armour plate (CAD, aluminium)": plate.volume * RHO_AL,
                          "legs 4x (CAD)": 4 * leg.volume * RHO_PA12CF, "leg servos 12x (hip yaw, hip pitch, knee per leg)": 12 * ACT["leg"]["mass_g"], "joint actuators 2x": 2 * ACT["joint"]["mass_g"], "MCU, IMU, bus, sensors": 90.0, "wiring, pins, pads": 80.0})
MODULES = pd.DataFrame([("camera", 180), ("IR sensor", 220), ("lidar", 350), ("IMU", 60), ("env sensor", 120), ("fibre optic spool", 900), ("battery (Li-ion 300 Wh)", 1800), ("AI core", 400), ("tool / sampler", 1200), ("medical", 800), ("odor module", 250), ("dummy / armour", 0)],
                       columns=["module", "payload_g"]).set_index("module")
MODULES["segment_g"] = base_segment.sum() + MODULES["payload_g"]
head_g = head.volume * RHO_PA12CF + 2 * 180 + 350 + 150 + 200 + ACT["joint"]["mass_g"]   # stereo cameras, lidar, illuminators, computer, its joint half
CONFIGS = {"mini (4)": ["camera", "battery (Li-ion 300 Wh)", "AI core", "env sensor"],
           "standard (8)": ["camera", "IR sensor", "lidar", "battery (Li-ion 300 Wh)", "AI core", "env sensor", "battery (Li-ion 300 Wh)", "tool / sampler"],
           "max (14)": ["camera", "IR sensor", "lidar", "IMU", "battery (Li-ion 300 Wh)", "AI core", "env sensor", "fibre optic spool", "battery (Li-ion 300 Wh)", "tool / sampler", "medical", "odor module", "battery (Li-ion 300 Wh)", "dummy / armour"]}
G = 9.81
cfg = pd.DataFrame({k: {"segments": len(v), "mass [kg]": (head_g + MODULES.loc[v, "segment_g"].sum()) / 1000, "batteries [Wh]": 300 * v.count("battery (Li-ion 300 Wh)"),
                        "length walk [mm]": p["head_length"] + len(v) * aph.pitch_length(p), "ball diameter [mm]": 2 * aph.ball_radius(aph.resolve(n_segments=len(v)))} for k, v in CONFIGS.items()}).T
print(f"base segment {base_segment.sum():.0f} g before its module (the sheet's 0.8–1.0 kg is the module payload; its 15–50 kg full system is what the configurations below must land in)")
display(base_segment.round(0).to_frame("g")); display(MODULES); cfg.round(2)

## 2. The ball: geometry, rolling, a drop

The ball's diameter follows from the segment count — the ring's circumference is `(n + 1) × pitch`.
Rolling: a ball on a slope starts rolling when the slope exceeds the CG offset (the head and payloads are
not symmetric); on the flat the legs push it, up to the sheet's 15 cm/s. A drop in ball mode (deployment
from a carrier) lands on one plate: the fall's energy over the plate's allowed deflection gives the impact
force, which goes into the plate FEA below — and, scaled back, the drop height the plate allows.

In [ ]:
ns = np.arange(4, 21)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ns, [2 * aph.ball_radius(aph.resolve(n_segments=int(n))) for n in ns], "o-", label="ball diameter"); ax.plot(ns, [2 * aph.coil_radius(aph.resolve(n_segments=int(n))) - p["body_diameter"] for n in ns], "s--", label="hole in the middle of the ring")
ax.set(xlabel="segments", ylabel="mm", title="ball size vs segment count (150 mm segments, 18 mm joints)"); ax.legend(); ax.grid(alpha=0.3)
M_STD = cfg.loc["standard (8)", "mass [kg]"]; R_BALL = aph.ball_radius(p) / 1000
CG_OFFSET = 0.025                                                                          # the CG off the ball's centre [m] (input: head + payload asymmetry)
slope_start = math.degrees(math.asin(CG_OFFSET / R_BALL))
V_ROLL = 0.15
E_ROLL = 0.5 * M_STD * V_ROLL**2 * (1 + 0.6)                                             # translation + rotation (a thick ring: I ≈ 0.6 m R²)
DROP_M, DELTA_M = 0.5, 0.030                                                              # deployment drop and the crush of the foam liner under the plates [m] (inputs)
F_IMPACT = M_STD * G * (DROP_M / DELTA_M + 1)
F_IMPACT_PAD = F_IMPACT * math.pi / 2                                                     # half-sine peak
roll = pd.Series({"standard config [kg]": M_STD, "ball diameter [m]": 2 * R_BALL, "slope to start rolling [deg]": slope_start, "kinetic energy at 15 cm/s [J]": E_ROLL,
                  "rolling 10 m downhill at 5° — speed [m/s]": math.sqrt(2 * G * 10 * math.sin(math.radians(5)) / 1.6), f"drop {DROP_M} m: impact force [N]": F_IMPACT, "impact peak on the plate pad [N]": F_IMPACT_PAD,
                  "impact as g": F_IMPACT_PAD / (M_STD * G)})
roll.to_frame("value").round(2)

## 3. Curling up: the joints

Curling from a walk means lifting the chain off the ground joint by joint. The worst moment sits at the
middle joint while it folds half the chain: the weight of `k` segments at their lever arms. The joint
actuator holds that moment with a margin; the two-axis joint's pin takes the same moment as a couple over
the neck.

In [ ]:
PITCH_M = aph.pitch_length(p) / 1000
W_SEG = MODULES.loc["camera", "segment_g"] / 1000 * G                                      # a typical segment with a light module
def curl_moment(k):                                                                        # k segments hanging straight out from a joint
    return sum(W_SEG * (PITCH_M * (i + 0.5)) for i in range(k))
ks = np.arange(1, N + 1)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ks, [curl_moment(int(k)) for k in ks], "o-", label="moment at the joint lifting k segments")
ax.axhline(ACT["joint"]["stall_Nm"], color="k", ls="-.", label=f"joint actuator {ACT['joint']['stall_Nm']:.0f} Nm"); ax.axhline(ACT["joint"]["stall_Nm"] / 1.5, color="#c62828", ls="--", label="/ SF 1.5")
ax.set(xlabel="segments lifted", ylabel="N m", title="curl-up: joint moment"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
K_MAX = int(max(k for k in ks if curl_moment(int(k)) * 1.5 <= ACT["joint"]["stall_Nm"]))
M_CURL = curl_moment(K_MAX)
print(f"a joint can lift {K_MAX} segments straight out (of {N}); curling the ring needs at most half the chain lifted at a time: {curl_moment(N // 2):.0f} Nm, SF {ACT['joint']['stall_Nm'] / curl_moment(N // 2):.2f}")

## 4. Walking: foot forces and the leg servo

Each segment stands on its own four legs. The leg-track crawl (wave gait, duty β = 0.8) at the sheet's
6 cm/s, a brisk walk (β = 0.5) and a step down a 0.3 m kerb onto one segment's legs. Half-sine stance
peaks, the hip servo over the horizontal reach, the knee over the tibia lever.

In [ ]:
W_HEAVY = MODULES.loc["battery (Li-ion 300 Wh)", "segment_g"] / 1000 * G
def foot_peak(Wseg, beta, legs=4): return (math.pi / 2) * Wseg / (legs * beta)
def leg_torques(Fz, Fx):
    a, b = math.radians(p["hip_angle_deg"]), math.radians(p["knee_angle_deg"])
    y_knee = p["femur_length"] * math.cos(a); y_foot = y_knee + p["tibia_length"] * math.cos(b); z_foot = REACH["drop_z"]
    return {"tau_hip_Nm": Fz * y_foot / 1000, "tau_knee_Nm": Fz * (y_foot - y_knee) / 1000, "tau_long_Nm": Fx * z_foot / 1000}
S_LEG = 0.05
F_KERB = W_HEAVY * (0.3 / S_LEG + 1) / 4 * math.pi / 2
gaits = {"crawl 6 cm/s (β 0.8)": foot_peak(W_HEAVY, 0.8), "walk (β 0.5)": foot_peak(W_HEAVY, 0.5), "three legs after a loss, crawl": foot_peak(W_HEAVY, 0.8 * 4 / 3, legs=3), "kerb 0.3 m down": F_KERB}
gt = pd.DataFrame({k: {"peak foot [N]": F, **leg_torques(F, 0.3 * F)} for k, F in gaits.items()}).T
gt["servo SF"] = ACT["leg"]["stall_Nm"] / gt[["tau_hip_Nm", "tau_knee_Nm", "tau_long_Nm"]].max(axis=1)
print(f"battery segment {W_HEAVY / G:.2f} kg | leg servo {ACT['leg']['name']}")
gt.round(2)

## 5. Strength (Talos): plate, leg, joint neck

The segment with its plate, held at its two necks: the drop impact on the plate's crown pad, and the ball
standing on one plate under the whole standard configuration (someone steps on it: 2 × the weight). The
leg in its frame: the walk peak, the kerb, a stuck leg at servo stall. The neck: the curl-up moment as a
couple over the joint.

In [ ]:
PA12CF = talos.Material("PA12-CF (printed)", youngs_modulus=3500.0, poissons_ratio=0.40, density=1.1e-9, yield_strength=60.0, source="nominal datasheet, flat orientation")
AL6061 = talos.Material("aluminium 6061-T6", youngs_modulus=69000.0, poissons_ratio=0.33, density=2.7e-9, yield_strength=240.0, source="handbook")
Ls, D, g_, dj = p["seg_length"], p["body_diameter"], p["joint_gap"], p["joint_diameter"]
SEG_REGIONS = [talos.SurfacesInBox("front_neck", (Ls / 2 + 0.5, -dj / 2 - 1, -dj / 2 - 1, Ls / 2 + g_ / 2 + 4, dj / 2 + 1, dj / 2 + 1)),
               talos.SurfacesInBox("rear_neck", (-Ls / 2 - g_ / 2 - 4, -dj / 2 - 1, -dj / 2 - 1, -Ls / 2 - 0.5, dj / 2 + 1, dj / 2 + 1)),
               talos.SurfacesOnPlane("pad", "z", aph.pad_z(p))]
def seg_model(supports, loads, name, material=PA12CF):
    return talos.StructuralModel(cad["segment"].artifacts["step"], "mm-N-MPa", material, SEG_REGIONS, supports, loads, talos.MeshSettings(element_size=6.0, min_element_size=1.5, high_order_optimize=2), name=name)   # elastic high-order optimisation: the default pass fails on the spherical plate
seg_model([talos.FixedSupport("front_neck")], [talos.Force("pad", fz=-1.0)], "mesh").mesh(RUNS / "seg_mesh", progress=True)
BOTH = [talos.FixedSupport("front_neck"), talos.FixedSupport("rear_neck")]
SEG_CASES = {
    "drop in ball mode: impact on the plate pad": (BOTH, [talos.Force("pad", fz=-F_IMPACT_PAD)]),
    "stepped on: 2 x the standard config on one plate": (BOTH, [talos.Force("pad", fz=-2 * M_STD * G)]),
    "curl-up: half the chain on the rear neck": ([talos.FixedSupport("front_neck")], [talos.Force("rear_neck", fz=-curl_moment(N // 2) * 1000 / (Ls / 2 + g_ / 2))]),
}
seg_results = {}
for name, (sup, loads) in tqdm(SEG_CASES.items()):
    key = "seg_" + name.split(":")[0].replace(" ", "_").replace("-", "_")
    mdl = seg_model(sup, loads, key); mdl.mesh(RUNS / key); seg_results[name] = mdl.solve(RUNS / key)
print({k: r.status for k, r in seg_results.items()})

In [ ]:
fy, dz, fd = REACH["foot_y"], REACH["drop_z"], p["foot_diameter"]
LEG_REGIONS = [talos.SurfacesInBox("hip", (-12, -6.0, -6.0, 12, 6.0, 6.0)), talos.SurfacesInBox("foot", (-fd, fy - fd, -dz - fd, fd, fy + fd, -dz + fd))]
def leg_model(loads, name):
    return talos.StructuralModel(cad["leg"].artifacts["step"], "mm-N-MPa", PA12CF, LEG_REGIONS, [talos.FixedSupport("hip")], loads, talos.MeshSettings(element_size=3.5, min_element_size=1.2, curvature_points=10), name=name)
F_STALL = ACT["leg"]["stall_Nm"] * 1000 / REACH["foot_y"]
LEG_CASES = {"walk peak": dict(fz=gaits["walk (β 0.5)"], fx=0.3 * gaits["walk (β 0.5)"]), "kerb 0.3 m down": dict(fz=F_KERB, fx=0.2 * F_KERB), "stuck leg at servo stall": dict(fy=-F_STALL, fz=0.5 * F_STALL)}
leg_model([talos.Force("foot", fz=1.0)], "mesh").mesh(RUNS / "leg_mesh", progress=True)
leg_results = {}
for name, f in tqdm(LEG_CASES.items()):
    key = "leg_" + name.replace(" ", "_").replace(".", "")
    mdl = leg_model([talos.Force("foot", **f)], key); mdl.mesh(RUNS / key); leg_results[name] = mdl.solve(RUNS / key)
def rows(results, label):
    return pd.DataFrame({f"{label} — {k}": {"max_von_mises_MPa": r.metrics.get("max_von_mises"), "SF_yield": r.metrics.get("safety_factor_yield"), "deflection_mm": r.metrics.get("max_displacement")} for k, r in results.items()}).T
static = pd.concat([rows(seg_results, "segment"), rows(leg_results, "leg")])
fig, ax = plt.subplots(figsize=(8, 4))
static["SF_yield"].astype(float).plot.barh(ax=ax, color=["#2e7d32"] * len(SEG_CASES) + ["#1565c0"] * len(LEG_CASES)); ax.axvline(1.5, color="#c62828", ls="--", label="SF 1.5"); ax.set(xscale="log", xlabel="safety factor on yield (the shell and plate as printed PA12-CF)"); ax.legend(); ax.grid(alpha=0.3, axis="x")
display(static.round(2))
tviz.show(tviz.plot_results(seg_results["drop in ball mode: impact on the plate pad"], field="von_mises"))

The plate is analysed as printed PA12-CF like the rest of the shell; the mass budget already assumes an
aluminium plate. If the drop case is the one below SF 1.5, the aluminium plate (yield 240 MPa, four times
stiffer) is the fix — rerun the two plate cases with `material=AL6061`:

In [ ]:
al = {}
for name in ("drop in ball mode: impact on the plate pad", "stepped on: 2 x the standard config on one plate"):
    key = "seg_al_" + name.split(":")[0].replace(" ", "_").replace("-", "_")
    mdl = seg_model(*SEG_CASES[name], key, material=AL6061); mdl.mesh(RUNS / key); al[name] = mdl.solve(RUNS / key)
al_tab = rows(al, "segment, aluminium")
sf_drop = float(al_tab.iloc[0]["SF_yield"])
# the impact force scales with (h / δ + 1): the drop height at which the aluminium plate reaches SF 1.5
F_allow = F_IMPACT_PAD * sf_drop / 1.5
H_ALLOW = (F_allow / (math.pi / 2) / (M_STD * G) - 1) * DELTA_M
print(f"aluminium plate: SF {sf_drop:.2f} at a {DROP_M} m drop -> allowable deployment drop {max(H_ALLOW, 0):.2f} m onto the {DELTA_M * 1000:.0f} mm liner (SF 1.5)")
al_tab.round(2)

## 6. Endurance per mode

Battery modules of 300 Wh. The leg-track crawl from a cost of transport, the sentinel from the sensors
and the holding servos, hibernation in the ball from the MCUs alone (the sheet says 12–72 h).

In [ ]:
COT = 2.0
rows_e = {}
for cname, mods in CONFIGS.items():
    m_kg = cfg.loc[cname, "mass [kg]"]; E = cfg.loc[cname, "batteries [Wh]"] * 0.9
    modes_p = {"crawl 6 cm/s": 25 + len(mods) * 4 * 0.8 + COT * m_kg * G * 0.06 / 0.5, "sentinel": 18 + len(mods) * 1.2, "ball hibernation": 3 + len(mods) * 0.3}
    for mode, P in modes_p.items():
        rows_e[f"{cname}, {mode}"] = {"power [W]": P, "hours": E / P}
pd.DataFrame(rows_e).T.round(1)

In [ ]:
summary_doc = {"family": "Myropods", "member": "Apheloria", "design": {"file": "designs/apheloria.py", "parameters": p}, "configs": cfg.to_dict(orient="index"), "modules": MODULES.to_dict(orient="index"),
               "ball": {k: float(v) for k, v in roll.items()}, "curl": {"k_max_lifted": K_MAX, "moment_half_chain_Nm": curl_moment(N // 2)}, "gaits": gt.to_dict(orient="index"),
               "static": static.to_dict(orient="index"), "static_aluminium_plate": al_tab.to_dict(orient="index"), "allowable_drop_m": H_ALLOW, "endurance": rows_e}
(RUNS / "apheloria.json").write_text(json.dumps(summary_doc, indent=2, default=str))
print("written:", sorted(x.name for x in RUNS.iterdir()))

**Next steps an engineer would take:** the plates close the ball over the back but leave the ring's sides
open — a wider plate or hinged side caps on the head and tail segment are the next CAD revision; the
curl-up moment sizes the joint actuator (a 60 Nm quasi-direct-drive is heavy — a lighter one with a
locking brake holds the ball for nothing); the kerb step-down is the leg servo's limiting case (SF < 1 at
full speed: descend slowly, legs compliant, or the next servo class); a module's locking latch, not
modelled here, carries the same curl moment and needs its own FEA; and the deployment drop is bounded by
the plate — a thicker foam liner buys drop height linearly.